In [1]:
import pandas as pd
import os
from pathlib import Path

print("Keeps articles with **25–500** words")

# ────────────────────────────────────────────────
# CONFIG
# ────────────────────────────────────────────────

paths = {
    "Real": r"H:\\data_cset312\\real_cleaned.csv",
    "Fake": r"H:\\data_cset312\\fake_cleaned.csv"
}

output_paths = {
    "Real": r"H:\\data_cset312\\real_filtered_25-500w.csv",
    "Fake": r"H:\\data_cset312\\fake_filtered_25-500w.csv"
}

TEXT_COLUMN   = "text"           # ← change if your column is named differently (article, content, body, news, ...)
MIN_WORDS     = 25               # ← changed here as requested
MAX_WORDS     = 500
CHUNKSIZE     = 50000            # lower to 20000 or 10000 if memory is still tight
WRITE_HEADER  = True

# ────────────────────────────────────────────────

def filter_file_chunked(input_path, output_path, name):
    if not Path(input_path).is_file():
        print(f"File not found: {input_path}")
        return 0, 0

    print(f"┌─ Processing {name}: {os.path.basename(input_path)}")
    print(f"   Reading in chunks of {CHUNKSIZE:,} rows...\n")

    total_processed = 0
    total_kept      = 0

    # Delete old output file if it exists
    if Path(output_path).exists():
        Path(output_path).unlink()

    try:
        chunks = pd.read_csv(
            input_path,
            chunksize=CHUNKSIZE,
            usecols=[TEXT_COLUMN],          # memory optimization
            low_memory=False,
            encoding='utf-8',
            on_bad_lines='skip',
            dtype={TEXT_COLUMN: 'string'}
        )
    except Exception as e:
        print(f"Error opening file: {e}")
        return 0, 0

    first_chunk = True

    for i, chunk in enumerate(chunks, 1):
        chunk_size = len(chunk)
        total_processed += chunk_size

        # Clean text & count words
        chunk[TEXT_COLUMN] = chunk[TEXT_COLUMN].astype(str).fillna('').str.strip()
        chunk['word_count'] = chunk[TEXT_COLUMN].str.split().str.len()

        # Filter
        mask = (chunk['word_count'] >= MIN_WORDS) & (chunk['word_count'] <= MAX_WORDS)
        kept_chunk = chunk[mask][[TEXT_COLUMN]].copy()
        kept_chunk['label'] = 0 if name == "Real" else 1

        kept_in_this_chunk = len(kept_chunk)

        # Append to CSV
        mode = 'w' if first_chunk else 'a'
        header = first_chunk and WRITE_HEADER
        kept_chunk.to_csv(
            output_path,
            mode=mode,
            header=header,
            index=False,
            encoding='utf-8'
        )

        total_kept += kept_in_this_chunk
        removed_in_chunk = chunk_size - kept_in_this_chunk

        print(f"   Chunk {i:>3} | processed: {chunk_size:>8,} | kept: {kept_in_this_chunk:>7,} | removed: {removed_in_chunk:>7,}")

        first_chunk = False

    removed_total = total_processed - total_kept
    keep_pct = (total_kept / total_processed * 100) if total_processed > 0 else 0

    print(f"\n   FINAL ───────────────────────────────────")
    print(f"   Original rows     : {total_processed:>12,}")
    print(f"   Removed           : {removed_total:>12,}")
    print(f"   Kept (25–500 words): {total_kept:>12,}")
    print(f"   Keep ratio        : {keep_pct:>6.1f}%")
    print(f"   → Saved to: {os.path.basename(output_path)}")
    print("└───────────────────────────────────────────────\n")

    return total_processed, total_kept


# ────────────────────────────────────────────────
# RUN BOTH FILES
# ────────────────────────────────────────────────

stats = {}

for cls in ["Real", "Fake"]:
    orig, kept = filter_file_chunked(
        paths[cls],
        output_paths[cls],
        cls
    )
    stats[cls] = {"original": orig, "kept": kept}

print("Summary ───────────────────────────────────────")
for cls, numbers in stats.items():
    if numbers['original'] > 0:
        print(f"{cls:>6} → original: {numbers['original']:>10,} | kept: {numbers['kept']:>10,} "
              f"({numbers['kept']/numbers['original']*100:5.1f}%)")
    else:
        print(f"{cls:>6} → file not processed or empty")

print("\nDone. Filtered files created with min 25 words per article.")
print("You should now see slightly more rows kept compared to min=30.")

Keeps articles with **25–500** words
┌─ Processing Real: real_cleaned.csv
   Reading in chunks of 50,000 rows...

   Chunk   1 | processed:   50,000 | kept:  30,048 | removed:  19,952
   Chunk   2 | processed:   50,000 | kept:  30,121 | removed:  19,879
   Chunk   3 | processed:   50,000 | kept:  30,068 | removed:  19,932
   Chunk   4 | processed:   50,000 | kept:  30,201 | removed:  19,799
   Chunk   5 | processed:   50,000 | kept:  30,029 | removed:  19,971
   Chunk   6 | processed:   50,000 | kept:  30,070 | removed:  19,930
   Chunk   7 | processed:   50,000 | kept:  30,098 | removed:  19,902
   Chunk   8 | processed:   50,000 | kept:  30,197 | removed:  19,803
   Chunk   9 | processed:   50,000 | kept:  30,245 | removed:  19,755
   Chunk  10 | processed:   50,000 | kept:  29,982 | removed:  20,018
   Chunk  11 | processed:   50,000 | kept:  30,188 | removed:  19,812
   Chunk  12 | processed:   50,000 | kept:  30,277 | removed:  19,723

   FINAL ───────────────────────────────────


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc

print("=== Subsampling to ~300k rows per class while trying to preserve word-count mean & std ===\n")

# ────────────────────────────────────────────────
# CONFIGURATION
# ────────────────────────────────────────────────

INPUT_FILES = {
    "Real": r"H:\\data_cset312\\real_filtered_25-500w.csv",
    "Fake": r"H:\\data_cset312\\fake_filtered_25-500w.csv"
}

OUTPUT_FILES = {
    "Real": r"H:\\data_cset312\\real_300k_matched.csv",
    "Fake": r"H:\\data_cset312\\fake_300k_matched.csv"
}

TARGET_N      = 300_000
TEXT_COL      = "text"
CHUNKSIZE     = 60_000           # lower to 30_000–40_000 if memory is very tight
SEED          = 42
TOLERANCE     = 2.0              # acceptable deviation in words (mean / std)

np.random.seed(SEED)

# ────────────────────────────────────────────────

def compute_wordcount_stats(path):
    """Estimate mean & std of word counts using chunks (low memory)"""
    if not Path(path).is_file():
        raise FileNotFoundError(path)

    print(f"Computing stats for {Path(path).name} ...")
    
    total_words = 0
    total_count = 0
    sum_sq      = 0
    n_rows      = 0

    chunks = pd.read_csv(
        path,
        usecols=[TEXT_COL],
        chunksize=CHUNKSIZE,
        dtype={TEXT_COL: 'string'},
        on_bad_lines='skip'
    )

    for chunk in chunks:
        chunk = chunk.dropna(subset=[TEXT_COL])
        if len(chunk) == 0:
            continue
            
        wc = chunk[TEXT_COL].str.split().str.len()
        wc = wc[(wc >= 25) & (wc <= 500)]
        
        n = len(wc)
        if n == 0:
            continue
            
        total_words += wc.sum()
        total_count += n
        sum_sq      += (wc ** 2).sum()
        n_rows      += n

        print(f"  processed rows: {n_rows:,}", end='\r')

    if total_count == 0:
        raise ValueError("No valid rows found")

    mean = total_words / total_count
    variance = (sum_sq / total_count) - mean**2
    std = np.sqrt(variance) if variance > 0 else 0

    print(f"\n  rows used: {n_rows:,}")
    print(f"  mean word count: {mean:.2f}")
    print(f"  std  word count: {std:.2f}\n")
    
    return mean, std, n_rows


def subsample_close_to_moments(
    input_path, output_path, target_n, ref_mean, ref_std, label
):
    """Two-pass subsampling: stratified on word-count bins + small correction"""
    print(f"Subsampling {Path(input_path).name} → ~{target_n:,} rows")

    if not Path(input_path).is_file():
        print("  File not found → skipping")
        return 0, 0, 0

    # ─── PASS 1: Collect word counts and valid row indices ──────────────────
    print("  Pass 1: collecting word counts & indices ...")
    
    wc_list  = []
    indices  = []
    offset   = 0

    chunks = pd.read_csv(
        input_path,
        usecols=[TEXT_COL],
        chunksize=CHUNKSIZE,
        dtype={TEXT_COL: 'string'}
    )

    for chunk_i, chunk in enumerate(chunks, 1):
        chunk_wc = chunk[TEXT_COL].astype(str).str.split().str.len()
        valid    = (chunk_wc >= 25) & (chunk_wc <= 500)
        
        wc_list.extend(chunk_wc[valid].values)

        # FIXED: safe index selection
        chunk_indices = range(offset, offset + len(chunk))
        selected_idx  = [idx for idx, keep in zip(chunk_indices, valid) if keep]
        indices.extend(selected_idx)
        
        offset += len(chunk)
        print(f"  chunk {chunk_i} done  ({len(wc_list):,} valid so far)", end='\r')

    print()  # newline after progress

    if len(wc_list) <= target_n:
        print("  Fewer rows than target → copying entire file")
        df_full = pd.read_csv(input_path)
        df_full['label'] = label
        df_full.to_csv(output_path, index=False)
        m = np.mean(wc_list)
        s = np.std(wc_list)
        return len(wc_list), m, s

    wc_arr = np.array(wc_list)
    del wc_list
    gc.collect()

    print(f"  Total valid rows: {len(wc_arr):,}")

    # ─── PASS 2: Stratified sampling by word-count quantiles ────────────────
    print("  Pass 2: stratified sampling ...")
    
    quantiles = np.quantile(wc_arr, np.linspace(0, 1, 11))  # 10 bins
    bin_indices = np.digitize(wc_arr, quantiles) - 1
    bin_indices = np.clip(bin_indices, 0, 9)

    selected = []

    for b in range(10):
        mask = bin_indices == b
        if not mask.any():
            continue
            
        cand_idx = np.where(mask)[0]
        # proportional allocation
        n_take = min(int(target_n * (mask.sum() / len(wc_arr))) + 1, mask.sum())
        chosen = np.random.choice(cand_idx, size=n_take, replace=False)
        selected.extend(indices[i] for i in chosen)

    # Supplement if needed
    if len(selected) < target_n:
        remaining = target_n - len(selected)
        all_indices_set = set(indices)
        selected_set = set(selected)
        available = [i for i in all_indices_set if i not in selected_set]
        extra = np.random.choice(available, size=min(remaining, len(available)), replace=False)
        selected.extend(extra)

    selected = sorted(selected)  # for sequential reading

    # ─── Final read of selected rows ────────────────────────────────────────
    print(f"  Reading {len(selected):,} selected rows ...")
    
    # skiprows expects 0-based line numbers, header is line 0
    df = pd.read_csv(input_path, skiprows=lambda x: x not in selected and x != 0)
    df['label'] = label
    df = df[[TEXT_COL, 'label']]

    current_wc = df[TEXT_COL].astype(str).str.split().str.len()
    current_mean = current_wc.mean()
    current_std  = current_wc.std()

    print(f"  Initial sample → mean: {current_mean:.2f}  std: {current_std:.2f}")

    # Light correction if deviation is too large
    if abs(current_mean - ref_mean) > TOLERANCE or abs(current_std - ref_std) > TOLERANCE:
        print("  Applying small length correction ...")
        df['wc'] = current_wc
        
        trim_fraction = 0.02  # trim ~2% from one tail
        
        if current_mean > ref_mean + TOLERANCE:
            # remove some longest
            df = df.sort_values('wc').iloc[:-int(len(df) * trim_fraction)].copy()
        elif current_mean < ref_mean - TOLERANCE:
            # remove some shortest
            df = df.sort_values('wc', ascending=False).iloc[:-int(len(df) * trim_fraction)].copy()

        current_wc   = df['wc']
        current_mean = current_wc.mean()
        current_std  = current_wc.std()
        df = df.drop(columns=['wc'])

    df.to_csv(output_path, index=False)
    print(f"  Final → {len(df):,} rows | mean: {current_mean:.2f} | std: {current_std:.2f}")
    print(f"  Saved → {Path(output_path).name}\n")

    return len(df), current_mean, current_std


# ────────────────────────────────────────────────
# MAIN EXECUTION
# ────────────────────────────────────────────────

stats = {}

for cls in ["Real", "Fake"]:
    mean, std, total = compute_wordcount_stats(INPUT_FILES[cls])
    stats[cls] = {"mean": mean, "std": std, "total": total}

print("Reference distributions:")
print(f"Real : mean {stats['Real']['mean']:.1f} ± {stats['Real']['std']:.1f}  (n={stats['Real']['total']:,})")
print(f"Fake : mean {stats['Fake']['mean']:.1f} ± {stats['Fake']['std']:.1f}  (n={stats['Fake']['total']:,})\n")

# Subsample both classes
results = {}

for cls in ["Real", "Fake"]:
    n, m, s = subsample_close_to_moments(
        INPUT_FILES[cls],
        OUTPUT_FILES[cls],
        TARGET_N,
        stats[cls]["mean"],
        stats[cls]["std"],
        label = 0 if cls == "Real" else 1
    )
    results[cls] = {"rows": n, "mean": m, "std": s}

print("═" * 70)
print("FINAL RESULT")
for cls in ["Real", "Fake"]:
    r = results[cls]
    diff_mean = r["mean"] - stats[cls]["mean"]
    diff_std  = r["std"]  - stats[cls]["std"]
    print(f"{cls:>6}: {r['rows']:,} rows | "
          f"mean {r['mean']:.1f} ({diff_mean:+.1f}) | "
          f"std {r['std']:.1f} ({diff_std:+.1f})")
print("═" * 70)

print("\nDone.")
print("You should now have two balanced ~300k files with reasonably matched length distributions.")

=== Subsampling to ~300k rows per class while trying to preserve word-count mean & std ===

Computing stats for real_filtered_25-500w.csv ...
  processed rows: 361,524
  rows used: 361,524
  mean word count: 212.09
  std  word count: 129.22

Computing stats for fake_filtered_25-500w.csv ...
  processed rows: 418,522
  rows used: 418,522
  mean word count: 201.90
  std  word count: 133.29

Reference distributions:
Real : mean 212.1 ± 129.2  (n=361,524)
Fake : mean 201.9 ± 133.3  (n=418,522)

Subsampling real_filtered_25-500w.csv → ~300,000 rows
  Pass 1: collecting word counts & indices ...
  chunk 7 done  (361,524 valid so far)
  Total valid rows: 361,524
  Pass 2: stratified sampling ...
  Reading 300,003 selected rows ...
  Initial sample → mean: 212.06  std: 129.16
  Final → 300,002 rows | mean: 212.06 | std: 129.16
  Saved → real_300k_matched.csv

Subsampling fake_filtered_25-500w.csv → ~300,000 rows
  Pass 1: collecting word counts & indices ...
  chunk 7 done  (418,522 valid so f